<a href="https://colab.research.google.com/github/sabujcb/Mycodes/blob/main/ex9_word2vec.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install gensim

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 49.7 MB/s eta 0:00:00


In [2]:
!wget http://dl.turkunlp.org/intro-to-nlp/enwiki-20220301-sample-tokenized.txt

--2026-04-22 18:20:12--  http://dl.turkunlp.org/intro-to-nlp/enwiki-20220301-sample-tokenized.txt
Resolving dl.turkunlp.org (dl.turkunlp.org)... 195.148.30.23
Connecting to dl.turkunlp.org (dl.turkunlp.org)|195.148.30.23|:80... connected.
HTTP request sent, awaiting response... 200 OK
Length: 543383248 (518M) [text/plain]
Saving to: ‘enwiki-20220301-sample-tokenized.txt’

enwiki-20220301-sam 100%[===================>] 518.21M  7.78MB/s    in 53s     

2026-04-22 18:21:06 (9.72 MB/s) - ‘enwiki-20220301-sample-tokenized.txt’ saved [543383248/543383248]



The dataset contains 3.8 million sentences and 102 million tokens. Training word2vec on full data and 5 epochs takes about 22 minutes. Trim the data while debugging.

In [3]:
# word2vec TOY EXAMPLE
# NOTE: This is a toy model that only demonstrates how to train word2vec,
# and that it will learn very simple patterns.
# This model will not give reasonable embedding similarities because of lack of data.

from gensim.models import Word2Vec

sentence_a = ["the", "cat", "liked", "milk"]
sentence_b = ["the", "dog", "ate", "food"]

sentences = [sentence_a]*7000 + [sentence_b]*7000 # repeat the same toy sentences multiple times

# train model
model = Word2Vec(
    sentences,
    sg=0,
    vector_size=40,
    window=2,
    min_count=1,
    workers=1,
    epochs=100
)

# test target word prediction (language modeling objective)
for context in [["the", "liked", "milk"], ["the", "ate", "food"], ["the", "liked", "food"]]:
    print("Context:", context)
    pred = model.predict_output_word(context, topn=5)
    print("Top predictions:")
    for w,s in pred:
        print(f"{w}: {s:.2f}")
    print()

# the model is able to learn a very simple pattern, where "liked milk" predicts "cat", and "ate food" predicts "dog"
# For "liked food" context, dog/cat/milk/ate are equally likely

Context: ['the', 'liked', 'milk']
Top predictions:
cat: 0.68
milk: 0.11
liked: 0.10
the: 0.06
dog: 0.02

Context: ['the', 'ate', 'food']
Top predictions:
dog: 0.72
food: 0.09
ate: 0.09
the: 0.06
cat: 0.01

Context: ['the', 'liked', 'food']
Top predictions:
dog: 0.22
milk: 0.21
cat: 0.20
ate: 0.20
the: 0.13



In [11]:
# STEP 1: Load dataset and lowercase tokens

file_path = "enwiki-20220301-sample-tokenized.txt"

sentences = []

with open(file_path, "r", encoding="utf-8") as f:
    for i, line in enumerate(f):
        tokens = line.strip().lower().split()
        if tokens:
            sentences.append(tokens)

        # final run: use much more data
        if i == 100000:   # try 100k first; if OK, go to 200k
            break

print("Total sentences loaded:", len(sentences))
print("Example sentence:", sentences[0])

Total sentences loaded: 100001
Example sentence: ['incumbent', 'cm', 'mayawati', 'began', 'her', 'campaign', 'on', '27', 'january', 'at', 'a', 'rally', 'in', 'bijnor', '.']


In [12]:
from gensim.models import Word2Vec

model = Word2Vec(
    sentences=sentences,
    vector_size=100,
    window=5,
    min_count=2,
    workers=4,
    sg=1,          # skip-gram often works better on smaller data
    epochs=10
)

print("Training complete!")

Training complete!


In [13]:
# STEP 3: Predict output word (language modeling)

contexts = [
    ["the", "cat", "is"],
    ["he", "was", "a"],
    ["this", "is", "very"],
]

for context in contexts:
    print("Context:", context)
    preds = model.predict_output_word(context, topn=5)

    if preds:
        for word, prob in preds:
            print(f"{word}: {prob:.4f}")
    else:
        print("No prediction (words may be OOV)")

    print()

Context: ['the', 'cat', 'is']
white: 0.0002
girl: 0.0001
song: 0.0001
character: 0.0001
story: 0.0001

Context: ['he', 'was', 'a']
member: 0.0003
elected: 0.0002
appointed: 0.0002
also: 0.0002
when: 0.0002

Context: ['this', 'is', 'very']
very: 0.0005
relatively: 0.0003
difficult: 0.0003
reason: 0.0003
unclear: 0.0003



In [14]:
# STEP 4: Similar words

words = ["good", "bad", "dog", "cat", "king", "queen"]

for word in words:
    print(f"\nTop similar words for '{word}':")

    try:
        similar = model.wv.most_similar(word, topn=5)
        for w, s in similar:
            print(f"{w}: {s:.4f}")
    except KeyError:
        print("Word not in vocabulary")


Top similar words for 'good':
laugh: 0.6709
worthwhile: 0.6682
really: 0.6567
proud: 0.6486
stupid: 0.6481

Top similar words for 'bad':
pretty: 0.7292
madness: 0.7048
myself: 0.6941
surprisingly: 0.6913
lovely: 0.6899

Top similar words for 'dog':
stump: 0.6890
appenzeller: 0.6880
purple: 0.6873
roses: 0.6789
sweet: 0.6782

Top similar words for 'cat':
blonde: 0.8030
goodbye: 0.7866
shirt: 0.7858
bright: 0.7800
brunette: 0.7795

Top similar words for 'king':
emperor: 0.6631
duke: 0.6609
anjou: 0.6540
kamehameha: 0.6517
tsar: 0.6513

Top similar words for 'queen':
elizabeth: 0.7096
coronation: 0.6800
1551: 0.6659
consort: 0.6631
boleyn: 0.6617
